In [ ]:
%load_ext autoreload
%autoreload 2
import dt4dds_benchmark
import plotly.express as px
import pandas as pd
import numpy as np

data_dropout = dt4dds_benchmark.analysis.Dataset.combine(*[dt4dds_benchmark.pipelines.HDF5Manager(f'./data/dropout/{s}.hdf5').get_data() for s in (
    'aeon_low', 'aeon_medium', 'aeon_high', 'rs_low', 'rs_medium', 'rs_high', 'modulation_medium', 'ldpc_medium', 'dbgps_low', 'dbgps_medium', 'dbgps_high',
)])
data_rate = dt4dds_benchmark.analysis.Dataset.combine(*[dt4dds_benchmark.pipelines.HDF5Manager(f'./data/rate/{s}.hdf5').get_data() for s in (
    'aeon_low', 'aeon_medium', 'aeon_high', 'rs_low', 'rs_medium', 'rs_high', 'modulation_medium', 'ldpc_medium', 'dbgps_low', 'dbgps_medium', 'dbgps_high',
)])

In [ ]:
colormap_type = {'DNAAeon': '#add1eb', 'DNARS': '#f9bd9f', 'DBGPS': '#31a354', 'LDPC': '#756bb1', 'Modulation': '#636363'}

In [ ]:
df_dropout = data_dropout.get_fits_by_group(['codec.type', 'codec.name', 'workflow.name', 'workflow.type', 'workflow.dropout'], 'workflow.overall_rate', additional_agg={'code_rate': 'mean'})
df_dropout['workflow.overall_rate'] = df_dropout['threshold']

df_rate = data_rate.get_fits_by_group(['codec.type', 'codec.name', 'workflow.name', 'workflow.type', 'workflow.overall_rate'], 'workflow.dropout', additional_agg={'code_rate': 'mean'})
df_rate['workflow.dropout'] = df_rate['threshold']

df = pd.concat([df_dropout, df_rate], ignore_index=True)

In [ ]:
# see https://stackoverflow.com/questions/32791911/fast-calculation-of-pareto-front-in-python
def is_pareto_optimal(costs):
    is_efficient = np.all(np.logical_not(np.isnan(costs)), axis=1)
    for i, c in enumerate(costs):
        if is_efficient[i]:
            is_efficient[is_efficient] = np.any(costs[is_efficient]<c, axis=1)  # Keep any point with a lower cost
            is_efficient[i] = True  # And keep self
    return is_efficient
    
# function that receives a groupby subset and only returns the pareto front
def apply_pareto(group, cost1, cost2, cost1_max = False, cost2_max = False):
    # get the costs
    costs = group[[cost1, cost2]].values.astype(float)
    # invert the costs if they are maximization problems
    if cost1_max:
        costs[:,0] = -costs[:,0]
    if cost2_max:
        costs[:,1] = -costs[:,1]
    # get the pareto front
    return group[is_pareto_optimal(costs)]

# function that receives a groupby subset and returns the complete pareto front by adding extreme points if not present already
def complete_pareto_front(group):
    full = group.copy()
    if full['workflow.overall_rate'].min() > 0.0001:
        add = pd.DataFrame.from_dict({'codec.type': full['codec.type'].iloc[0], 'codec.name': full['codec.name'].iloc[0], 'workflow.type': full['workflow.type'].iloc[0], 'workflow.overall_rate': 0.0001, 'workflow.dropout': 1.001*full['workflow.dropout'].max()}, orient='index').T
        full = pd.concat([full, add], ignore_index=True).reset_index(drop=True)
    if full['workflow.dropout'].min() > 0.001:
        add = pd.DataFrame.from_dict({'codec.type': full['codec.type'].iloc[0], 'codec.name': full['codec.name'].iloc[0], 'workflow.type': full['workflow.type'].iloc[0], 'workflow.overall_rate': 1.001*full['workflow.overall_rate'].max(), 'workflow.dropout': 0.001}, orient='index').T
        full = pd.concat([full, add], ignore_index=True).reset_index(drop=True)
    return full.reset_index(drop=True)

In [ ]:
idf = df.groupby(['codec.type', 'codec.name', 'workflow.type']).apply(apply_pareto, 'workflow.overall_rate', 'workflow.dropout', cost1_max = True, cost2_max = True, include_groups=False).reset_index()

In [ ]:
plotdf = idf.groupby(['codec.type', 'codec.name', 'workflow.type'])[['codec.type', 'codec.name', 'workflow.type', 'workflow.overall_rate', 'workflow.dropout']].apply(complete_pareto_front).reset_index(drop=True)

plotdf = plotdf.sort_values(['codec.type', 'workflow.overall_rate', 'workflow.dropout'], ascending=[False, True, True])
plotdf['codec.name'] = plotdf['codec.name'].replace({'high': 'High', 'medium': 'Medium', 'low': 'Low', 'default': 'Low'})
plotdf['name'] = plotdf['codec.type'] + '-' + plotdf['codec.name']
plotdf

In [ ]:
fig = px.line(
    plotdf,
    x='workflow.overall_rate', 
    y='workflow.dropout', 
    log_y=True, 
    # log_x=True, 
    color='codec.type',
    facet_col='codec.name',
    facet_col_spacing=0.08,
    markers=True,
    color_discrete_map=colormap_type,
    category_orders={'codec.name': ['High', 'Medium', 'Low'],},
    range_x=[0.0, 0.15],
    range_y=[0.005, 1],
)
fig.update_layout(
    showlegend=False,
    width=340,
    height=140,
    margin=dict(l=0, r=10, t=20, b=0),
)
fig.add_vline(
    x=0.02,
    line_dash='dot',
    line_width=1,
    row=1,
)
fig.add_vline(
    x=0.0065,
    line_dash='dash',
    line_width=1,
    row=1,
)
fig.update_xaxes(dtick=0.05)
fig.update_yaxes(dtick=1)
fig.update_xaxes(title='Error rate per nt', tickformat=",.0%", row=1)
fig.update_yaxes(title='Sequence dropout', tickformat=",.0%", col=1)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)
fig.write_image('./figures/pareto_front.svg')
fig.show()

# export data
plotdf.to_csv('./figures/pareto_front.csv', index=False)